In [65]:
import os
import joblib
import pickle
import numpy as np
import pandas as pd
import polars as pl
import torch

import matplotlib.pyplot as plt
import seaborn as sns


import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import zscore
from tqdm import tqdm
from functools import partial


# home-grown
import utils as ut
import model as mod
import lazydata as lzdt

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
pd.options.display.max_columns = 60

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
df_mm, dict_colnames = ut.load_mm_dataset("med_seq")

In [35]:
df_mm[dict_colnames["l_colnames_single_zscale"]] = df_mm[dict_colnames["l_colnames_single_zscale"]].apply(zscore)

In [46]:
cols_subset = ["left_man", "left_woman", "left_boy", "left_girl", "left_oldman", "left_oldwoman",
    "right_man", "right_woman", "right_boy", "right_girl", "right_oldman", "right_oldwoman",
    "Eastern", "Southern"]
df_subset = df_mm[cols_subset + ["right_picked"]].copy()

In [47]:
df_subset["is_train"] = np.random.choice([True, False], df_subset.shape[0], p=[.8, .2])

In [85]:
X_train = df_subset.query("is_train").drop(columns=["is_train", "right_picked"]).head(100000).to_numpy()
X_test = df_subset.query("~is_train").drop(columns=["is_train", "right_picked"]).head(100000).to_numpy()
y_train = np.ravel(df_subset.query("is_train").drop(columns=cols_subset + ["is_train"]).head(100000).to_numpy())
y_test = np.ravel(df_subset.query("~is_train").drop(columns=cols_subset + ["is_train"]).head(100000).to_numpy())

In [92]:
X_train_noculture = df_subset.query("is_train").drop(columns=["is_train", "right_picked", "Eastern", "Southern"]).head(100000).to_numpy()
X_test_noculture = df_subset.query("~is_train").drop(columns=["is_train", "right_picked", "Eastern", "Southern"]).head(100000).to_numpy()

In [86]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.05, 0.1, 0.15],
    "max_depth": [2, 3, 4]
}

In [87]:
fit_models = True

In [88]:
model_full = GradientBoostingClassifier()

grid_full = GridSearchCV(
    estimator=model_full,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="neg_log_loss",  # or another metric
    n_jobs=-1            # use all CPU cores
)

if fit_models:
    grid_full.fit(X_train, y_train)
    joblib.dump(grid_full, "models/mm-sklearn/gridsearch-age-culture.pkl")
else:
    grid_full = joblib.load("models/mm-sklearn/gridsearch-age-culture.pkl")

In [89]:
best_model_full = grid_full.best_estimator_
y_test_pred = best_model_full.predict(X_test)
y_train_pred = best_model_full.predict(X_train)
df_train_preds_full = pd.DataFrame({"y_train_true":y_train, "y_train_pred":y_train_pred}) 
df_train_preds_full["is_correct"] = df_train_preds_full["y_train_true"] == df_train_preds_full["y_train_pred"]
df_test_preds_full = pd.DataFrame({"y_test_true":y_test, "y_test_pred":y_test_pred})
df_test_preds_full["is_correct"] = df_test_preds_full["y_test_true"] == df_test_preds_full["y_test_pred"]

In [97]:
print(
    "AGE OF SAVED/SACRIFICED PEOPLE AND CULTURE OF DM:\n",
    "train accuracy: ", np.round(df_train_preds_full["is_correct"].mean(), 3), 
    "\ndev accuracy: ", np.round(df_test_preds_full["is_correct"].mean(), 3)
)

AGE OF SAVED/SACRIFICED PEOPLE AND CULTURE OF DM:
 train accuracy:  0.621 
dev accuracy:  0.614


Now drop culture dummies

In [93]:
model_full_noculture = GradientBoostingClassifier()

grid_full_noculture = GridSearchCV(
    estimator=model_full_noculture,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="neg_log_loss",  # or another metric
    n_jobs=-1            # use all CPU cores
)

if fit_models:
    grid_full_noculture.fit(X_train_noculture, y_train)
    joblib.dump(grid_full_noculture, "models/mm-sklearn/gridsearch-age-noculture.pkl")
else:
    grid_full_noculture = joblib.load("models/mm-sklearn/gridsearch-age-noculture.pkl")

In [95]:
best_model_full_noculture = grid_full_noculture.best_estimator_
y_test_pred_noculture = best_model_full_noculture.predict(X_test_noculture)
y_train_pred_noculture = best_model_full_noculture.predict(X_train_noculture)
df_train_preds_full_noculture = pd.DataFrame({"y_train_true":y_train, "y_train_pred":y_train_pred_noculture}) 
df_train_preds_full_noculture["is_correct"] = df_train_preds_full_noculture["y_train_true"] == df_train_preds_full_noculture["y_train_pred"]
df_test_preds_full_noculture = pd.DataFrame({"y_test_true":y_test, "y_test_pred":y_test_pred_noculture})
df_test_preds_full_noculture["is_correct"] = df_test_preds_full_noculture["y_test_true"] == df_test_preds_full_noculture["y_test_pred"]

In [96]:
print(
    "ONLY AGE OF SAVED/SACRIFICED PEOPLE:\n",
    "train accuracy: ", np.round(df_train_preds_full_noculture["is_correct"].mean(), 3), 
    "\ndev accuracy: ", np.round(df_test_preds_full_noculture["is_correct"].mean(), 3)
)

ALL VALUES AND PROBABILITIES:
 train accuracy:  0.621 
dev accuracy:  0.614
